# Agent Loop with LangChain

The previous notebook built an agent loop from scratch. This notebook implements the same loop and the same use case using **LangChain**, which abstracts the provider-specific plumbing into a consistent interface.

**Use case:** Vendor due diligence. Same goal: gather a structured snapshot of "Arion Data Systems", but LangChain handles the message formatting, tool binding, and response parsing.



We will
- See how LangChain abstractions map to the raw loop steps
- Understand how `bind_tools`, `@tool`, and `invoke()` replace the manual boilerplate
- Recognise what LangChain hides and what it exposes
- Understand the trade-off: less boilerplate vs less visibility into provider behaviour


## Setup

In [ ]:
import os
from datetime import date
from langchain_core.messages import HumanMessage, AIMessage, ToolMessage, SystemMessage
from langchain_core.tools import tool
from langchain.chat_models import init_chat_model


llm = init_chat_model(model_provider='google_genai', model='gemini-2.5-flash-lite')

---
## The Task



In [10]:
TASK = (
    "Compile a due-diligence snapshot for vendor 'Arion Data Systems'. "
    "I need: founding year, approximate headcount, latest funding round "
    "(amount + date + lead investor), CEO name, reported ARR if available, "
    "and any legal or compliance red flags. Include today's date in the snapshot."
)
print("Task:", TASK)

Task: Compile a due-diligence snapshot for vendor 'Arion Data Systems'. I need: founding year, approximate headcount, latest funding round (amount + date + lead investor), CEO name, reported ARR if available, and any legal or compliance red flags. Include today's date in the snapshot.


---
## Step 1: Tools with `@tool`

LangChain's `@tool` decorator does two things at once:
1. Wraps the Python function so LangChain can call it
2. Extracts the schema automatically from the docstring and type hints

Compare this to earlier where we wrote `FunctionDeclaration + Schema` objects manually.

> **What `@tool` generates:** a `StructuredTool` with a name, description, and
> `args_schema` (a Pydantic model). LangChain serialises this into whatever format
> the provider requires — Gemini `FunctionDeclaration`, OpenAI function spec, etc.


In [11]:
@tool
def get_company_profile(company_name: str) -> str:
    """Return basic company profile: founding year, headcount, HQ, sector, key products."""
    if "arion" in company_name.lower():
        return (
            "Arion Data Systems | Founded: 2017 | Employees: ~340 | HQ: Amsterdam, NL | "
            "Sector: B2B SaaS -- data pipeline tooling for regulated industries | "
            "Products: ArcPipeline (ETL), ArcVault (compliance data store)"
        )
    return f"No profile found for '{company_name}'"


@tool
def get_funding_history(company_name: str) -> str:
    """Return known funding rounds with amounts, dates, and lead investors."""
    if "arion" in company_name.lower():
        return (
            "Series B: $28M (March 2023, Balderton Capital) | "
            "Series A: $9M (Oct 2020, Seedcamp) | "
            "Seed: $1.5M (2018) | Total raised: $38.5M | Status: Private"
        )
    return f"No funding data for '{company_name}'"


@tool
def search_news(company_name: str, topic: str) -> str:
    """Search recent news about a company on a specific topic such as 'CEO and leadership',
    'revenue and ARR', 'GDPR legal risk', or 'customer contracts'. One call per topic."""
    key = f"{company_name} {topic}".lower()
    if "arion" in key:
        if any(t in key for t in ("ceo", "leadership", "executive", "founder")):
            return (
                "CEO: Jonas Meier (co-founder, formerly Palantir). "
                "CTO: Priya Natarajan (joined 2021 from AWS). No recent changes."
            )
        if any(t in key for t in ("revenue", "arr", "financial", "growth")):
            return (
                "Reported ARR EUR 12M (Q4 2023 investor deck). NRR 118%. "
                "No audited public accounts (private company)."
            )
        if any(t in key for t in ("legal", "gdpr", "compliance", "risk", "red flag")):
            return (
                "Open GDPR complaint with Dutch DPA (Feb 2023). "
                "IP dispute with former engineer settled 2022. No other litigation."
            )
        if any(t in key for t in ("customer", "client", "partner")):
            return (
                "Customers: ING Bank (NL), Medicover (PL), unnamed UK insurer. "
                "Microsoft Azure partner. SOC 2 Type II certified."
            )
        return "No significant news for that topic."
    return f"No news for '{company_name}' on '{topic}'"


@tool
def get_current_date() -> str:
    """Return today's date in YYYY-MM-DD format."""
    return str(date.today())


TOOLS = [get_company_profile, get_funding_history, search_news, get_current_date]

# Inspect what @tool generated
for t in TOOLS:
    print(f"Tool: {t.name:30} | Args: {list(t.args.keys())}")


Tool: get_company_profile            | Args: ['company_name']
Tool: get_funding_history            | Args: ['company_name']
Tool: search_news                    | Args: ['company_name', 'topic']
Tool: get_current_date               | Args: []


---
## Step 2: Bind Tools to the Model

`bind_tools()` attaches the tool schemas to the model object.
When you call `llm_with_tools.invoke(messages)`, the model knows about the tools
and can request them in its response.

> Earlier, tools were passed to `GenerativeModel(tools=[...])` at creation time.
> Here, we bind them to an existing model instance. The result is the same.


In [12]:
llm_with_tools = llm.bind_tools(TOOLS)

SYSTEM_PROMPT = (
    "You are a corporate research assistant conducting vendor due diligence. "
    "Use the available tools to gather factual information. "
    "When you have enough information, produce a structured due-diligence snapshot."
)

TOOL_MAP = {t.name: t for t in TOOLS}
print("Tools bound:", [t.name for t in TOOLS])


Tools bound: ['get_company_profile', 'get_funding_history', 'search_news', 'get_current_date']


---
## Step 3: State

LangChain uses a list of typed `Message` objects as its state.

| LangChain type | Equivalent in raw loop |
|---|---|
| `HumanMessage` | `{"role": "user", "parts": [{"text": "..."}]}` |
| `AIMessage` | Model turn (may contain `tool_calls`) |
| `ToolMessage` | Tool result (keyed by `tool_call_id`) |
| `SystemMessage` | System instruction |

> **Note:** LangChain's `ToolMessage` uses a `tool_call_id` to match results to calls
> (similar to Anthropic, different from raw Gemini which uses function names).
> LangChain normalises this for you across providers.


In [13]:
messages = [
    SystemMessage(content=SYSTEM_PROMPT),
    HumanMessage(content=TASK),
]

print(f"Initial messages: {len(messages)}")
for m in messages:
    print(f"  [{m.__class__.__name__}]: {str(m.content)[:70]}")


Initial messages: 2
  [SystemMessage]: You are a corporate research assistant conducting vendor due diligence
  [HumanMessage]: Compile a due-diligence snapshot for vendor 'Arion Data Systems'. I ne


---
## Step 4: The Loop

With LangChain, the loop is more compact because message construction is handled by the framework. The conceptual steps are the same.

> **Comparison to barebones agent loop:**
> - `llm_with_tools.invoke(messages)` replaces `model.generate_content(contents=...)`
> - `response.tool_calls` replaces iterating over `response.candidates[0].content.parts`
> - `ToolMessage(content=..., tool_call_id=...)` replaces the `function_response` dict
> - `messages.append(response)` replaces storing `response.candidates[0].content`


In [14]:
MAX_STEPS = 10
step = 0
final_response = None

print("=" * 60)
print(f"GOAL: {TASK}")
print("=" * 60)

while step < MAX_STEPS:
    step += 1
    print(f"\n[Step {step}] Calling model...")

    # -- Reason ----------------------------------------------------------
    response = llm_with_tools.invoke(messages)
    messages.append(response)   # AIMessage (may contain tool_calls)

    # -- Decide ----------------------------------------------------------
    if not response.tool_calls:
        print(f"[Step {step}] Final answer received.")
        final_response = response
        break

    print(f"[Step {step}] Tool calls: {[tc['name'] for tc in response.tool_calls]}")

    # -- Act + Update ---------------------------------------------------
    for tc in response.tool_calls:
        tool_fn = TOOL_MAP[tc["name"]]
        result  = tool_fn.invoke(tc["args"])
        print(f"  {tc['name']}({tc['args']}) => {str(result)[:80]!r}...")
        messages.append(ToolMessage(content=str(result), tool_call_id=tc["id"]))

print("\n" + "=" * 60)
print("FINAL ANSWER:")
print(response.content)
print("=" * 60)


GOAL: Compile a due-diligence snapshot for vendor 'Arion Data Systems'. I need: founding year, approximate headcount, latest funding round (amount + date + lead investor), CEO name, reported ARR if available, and any legal or compliance red flags. Include today's date in the snapshot.

[Step 1] Calling model...
[Step 1] Tool calls: ['get_company_profile', 'get_funding_history', 'search_news', 'search_news', 'search_news', 'get_current_date']
  get_company_profile({'company_name': 'Arion Data Systems'}) => 'Arion Data Systems | Founded: 2017 | Employees: ~340 | HQ: Amsterdam, NL | Secto'...
  get_funding_history({'company_name': 'Arion Data Systems'}) => 'Series B: $28M (March 2023, Balderton Capital) | Series A: $9M (Oct 2020, Seedca'...
  search_news({'company_name': 'Arion Data Systems', 'topic': 'CEO and leadership'}) => 'CEO: Jonas Meier (co-founder, formerly Palantir). CTO: Priya Natarajan (joined 2'...
  search_news({'company_name': 'Arion Data Systems', 'topic': 'revenue and ARR

---
## Inspect the Message History

LangChain message objects are typed Python classes with consistent attributes,
making the conversation structure easy to introspect.


In [15]:
print(f"Total messages: {len(messages)}  |  Steps: {step}")
print()
for i, msg in enumerate(messages):
    cls = msg.__class__.__name__
    if hasattr(msg, "tool_calls") and msg.tool_calls:
        calls = [f"{tc['name']}({tc['args']})" for tc in msg.tool_calls]
        print(f"[{i}] {cls}: tool_calls = {calls}")
    elif hasattr(msg, "tool_call_id"):
        print(f"[{i}] {cls}: id={msg.tool_call_id} | {str(msg.content)[:80]}")
    else:
        print(f"[{i}] {cls}: {str(msg.content)[:100]}")


Total messages: 10  |  Steps: 2

[0] SystemMessage: You are a corporate research assistant conducting vendor due diligence. Use the available tools to g
[1] HumanMessage: Compile a due-diligence snapshot for vendor 'Arion Data Systems'. I need: founding year, approximate
[2] AIMessage: tool_calls = ["get_company_profile({'company_name': 'Arion Data Systems'})", "get_funding_history({'company_name': 'Arion Data Systems'})", "search_news({'company_name': 'Arion Data Systems', 'topic': 'CEO and leadership'})", "search_news({'company_name': 'Arion Data Systems', 'topic': 'revenue and ARR'})", "search_news({'company_name': 'Arion Data Systems', 'topic': 'GDPR legal risk'})", 'get_current_date({})']
[3] ToolMessage: id=28217b4c-fc5c-46e0-b6fb-7104286045ae | Arion Data Systems | Founded: 2017 | Employees: ~340 | HQ: Amsterdam, NL | Secto
[4] ToolMessage: id=2c72d1be-ca54-446f-8a35-eea52e8c016a | Series B: $28M (March 2023, Balderton Capital) | Series A: $9M (Oct 2020, Seedca
[5] ToolMessage: 

---
## LangChain vs Raw Loop

| Raw loop | LangChain|
|---|---|
| `FunctionDeclaration + Schema` objects | `@tool` decorator, schema auto-extracted |
| `genai.GenerativeModel(tools=[...])` | `llm.bind_tools(TOOLS)` |
| `model.generate_content(contents=...)` | `llm_with_tools.invoke(messages)` |
| Check `part.function_call` in parts | Check `response.tool_calls` |
| Mixed dict + protobuf message history | Typed `HumanMessage` / `AIMessage` / `ToolMessage` |
| `function_response` keyed by name | `ToolMessage(tool_call_id=...)` |
| ~70 lines of boilerplate | ~20 lines of loop code |

**What LangChain abstracts away:**
Provider-specific message formats, tool schema serialisation, response parsing,
and tool result formatting.

**What you lose:**
Direct visibility into the raw API request/response, fine-grained control over
message history structure, and understanding of provider-specific behaviour.


To summarise,

1. **LangChain collapses the three provider-specific functions** into a single `invoke()` call and a unified message format. This is its primary value proposition.

2. **`@tool` eliminates manual schema writing.** Well-written docstrings become better tool descriptions. The schema is inferred from type hints.

3. **`bind_tools()` makes tools portable across LLMs.** The same tool definitions work with Gemini, OpenAI, Anthropic, or any supported provider.

4. **The conceptual loop is unchanged.** Call model → check tool calls → execute → append results → repeat. LangChain reduces boilerplate but not the architecture.

5. **Typed message objects improve introspection.** `HumanMessage`, `AIMessage`, `ToolMessage` make the conversation structure explicit and easy to inspect.